<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/GEM_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [9]:
# @title Dependencies

# first-party
from typing import Dict, Any, Optional, Tuple, List

# third-party
from google.colab import drive
import pandas as pd
from openai import OpenAI


In [3]:
# @title Paths and definitions

# mount GDrive
drive.mount('/content/drive')

# dataset directory
DATASET_DIR = "/content/drive/MyDrive/Thesis/dataset/"

# raw directory
RAW_DIR = "/content/drive/MyDrive/Thesis/raw/"

# paper data + human reviews
DATASET_PAPER = "dataset_paper.parquet"

# llm reviews
LLM_REVIEW = 'llm_sim_results.csv'

# define log path
LOG_PATH = "/content/drive/MyDrive/Thesis/raw/standardise_progress.csv"

# define result path
RESULTS_PATH = "/content/drive/MyDrive/Thesis/raw/standardise_results.csv"

# system prompt for judgement processing
SYSTEM_PROMPT_PROCESSING = """

Carefully read the text of a scientific paper review. You should summarize each evaluation in the review in a separate line.
Begin each summary line with one of the following phrases: ’The reviewer appreciates’, ’The reviewer criticizes’, ’The reviewer questions’, ’The reviewer suggests’.
You need to keep the summary as concise as possible, excluding specific details about the paper’s content, such as topics, ideas, methods, findings, and any mathematical symbols.
You should ensure that even if multiple evaluations are mentioned in the same sentence in the original review, you should still split it into separate lines.
For example, you should not output a line like ’The reviewer appreciates the well-written paper and good experimental performance’.
In contrast, you should output ’The reviewer appreciates the well-written paper’ and ’The reviewer appreciates good experimental performance’ in two lines.

"""

OPENAI_KEY = "Set key here"

SIMULATE = True  # Set to False to make real API calls

MAX_RETRIES = 3

RETRY_DELAY = 2.0

PREPROCESSING_MODEL = "gpt-4o-2024-05-13"

Mounted at /content/drive


In [5]:
# @title Load ICLR data

# full path
FULL_PATH_ICLR = DATASET_DIR + DATASET_PAPER

try:
    # Load the Parquet file into a Pandas DataFrame
    df_reviews_source = pd.read_parquet(FULL_PATH_ICLR)

    # check
    print("\nDataFrame Columns:")
    print(df_reviews_source.columns.tolist())

# error message for file not found
except FileNotFoundError:
    print(f"ERROR: Parquet file not found at {FULL_PATH_ICLR}. Please check the path.")

# error message for file not readable
except Exception as e:
    print(f"An error occurred while reading the Parquet file: {e}")


DataFrame Head:


,paper_id,pdf_url,abstract,parsed_text,human_review1,human_review2,human_review3
0,vuD2xEtxZcj,https://openreview.net/pdf/b7b54047fdaf97f5057...,"In deep learning, fine-grained N:M sparsity re...",INTRODUCTION Pruning Deep Neural Networks (DNN...,Summary Of The Paper:\n\nThe paper aims to acc...,Summary Of The Paper:\n\nThis paper develops a...,Summary Of The Paper:\n\nThe paper presents a ...
1,WoByU5W5te0,https://openreview.net/pdf/e0646a302829ac0b05b...,We present a novel method to regularizes neura...,"INTRODUCTION Recently, representing a 3D scene...",Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focus on t...,Summary Of The Paper:\n\nThe paper proposes a ...
2,XZRmNjUMj0c,https://openreview.net/pdf/20516df845666b5c362...,Neural Architecture Search (NAS) is a fast-dev...,INTRODUCTION Neural Architecture Search (NAS) ...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThis paper focused on...,Summary Of The Paper:\n\nThe paper introduces ...
3,cDYRS5iZ16f,https://openreview.net/pdf/043fba8d0ed8251ba2e...,Scaling transformers has led to significant br...,INTRODUCTION The transformer architecture (Vas...,Summary Of The Paper:\n\nThis paper proposes a...,Summary Of The Paper:\n\nThe paper proposes a ...,Summary Of The Paper:\n\nThis paper investigat...
4,Sh97TNO5YY_,https://openreview.net/pdf/106dc37f05b949150cc...,We are interested in in silico evaluation meth...,INTRODUCTION Molecular optimization aims to di...,Summary Of The Paper:\n\nThis paper analyses r...,Summary Of The Paper:\n\nThe authors propose a...,"Summary Of The Paper:\n\nIn this work, the aut..."



DataFrame Columns:
['paper_id', 'pdf_url', 'abstract', 'parsed_text', 'human_review1', 'human_review2', 'human_review3']


In [ ]:
# @title Load LLM reviews

# full path
FULL_PATH_LLM = RAW_DIR + LLM_REVIEW

# adjust data types of directory
LLM_CSV_DTYPES = {
    'document_id': str,
    'review_text': str,
    'raw_api_output': str,
    'model': str,
    'temperature': float,
    'reasoning_effort': str,
    'chain_of_thought': str,
    'CoT': str
}

try:
    # Load the CSV file into a Pandas DataFrame
    df_llm_results = pd.read_csv(FULL_PATH_LLM, dtype=LLM_CSV_DTYPES)

    # check
    print("\nDataFrame Columns:")
    print(df_llm_results.columns)

# error message for file not found
except FileNotFoundError:
    print(f"ERROR: Parquet file not found at {FULL_PATH_LLM}. Please check the path.")

# error message for file not readable
except Exception as e:
    print(f"An error occurred while reading the Parquet file: {e}")


In [8]:
# @title Merging

try:
    # full-join merge
    df_merged_full = pd.merge(
        left=df_reviews_source,
        right=df_llm_results,
        left_on='paper_id',
        right_on='document_id',
        how='outer'
    )

    # check
    print(df_merged_full.columns.tolist())

# error message
except Exception as e:
    print(f"An error occured while merging the files: {e}")


['paper_id', 'pdf_url', 'abstract', 'parsed_text', 'human_review1', 'human_review2', 'human_review3', 'document_id', 'review_text', 'raw_api_output', 'model', 'temperature', 'reasoning_effort', 'chain_of_thought', 'CoT']


# Preprocessing

In [ ]:
# @title Define simulation

# define LLMClient
class LLMClient:
    """
    Handles API calls, simulation, and retry logic (adopted from supervisor's pattern).
    """
    def __init__(self, simulate: bool = True, api_key: str = OPENAI_KEY, seed: int = 7):
        self.simulate = simulate
        self.rng = random.Random(seed)
        self._openai = None
        self.api_key = api_key

        # Initialize client if not simulating
        self._maybe_init_clients()

    def _maybe_init_clients(self):
        """Initializes the real OpenAI client if not in simulation mode."""
        if self.simulate:
            print("INFO: LLMClient initialized in SIMULATION mode.")
            return

        if self.api_key == "SK-mock-key":
             print("WARNING: Using mock API key. Real API calls will likely fail. Please set OPENAI_API_KEY.")

        try:
            # We only need the OpenAI client for your current use case
            self._openai = OpenAI(api_key=self.api_key)
            print("INFO: LLMClient initialized for REAL API calls.")
        except Exception as e:
            print(f"ERROR: Could not initialize OpenAI client: {e}")
            self.simulate = True # Fallback to simulation if initialization fails

    def call(
        self,
        system_prompt: str,
        user_prompt: str,
        temperature: float,
        max_tokens: int,
        max_retries: int = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
        model: str = PREPROCESSING_MODEL
        ) -> Tuple[str, str]:
        """
        Executes the API call with retry logic (adopted from supervisor's pattern).

        Returns: (raw_text, status_code/error_message)
        """

        if self.simulate:
            # --- SIMULATION PATH (quick return) ---
            simulated_text = (
                f"Simulated Standardization: The core argument was '{user_prompt[:50]}...'. "
                f"Model={model}, Temp={temperature}. Output is a concise summary."
            )
            return simulated_text, "SIMULATED"

        # --- REAL API PATH (with retry loop) ---
        for attempt in range(1, max_retries + 1):
            try:
                # Direct implementation of your original function logic
                response = self._openai.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt}
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens
                )

                # Success: Extract the text and return
                generated_text = response.choices[0].message.content
                return generated_text, "SUCCESS"

            except Exception as e:
                error_msg = f"API Error (Attempt {attempt}/{max_retries}): {type(e).__name__}: {e}"

                if attempt == max_retries:
                    print(f"FATAL: {error_msg}")
                    return f"ERROR: Failed after {max_retries} attempts. {e}", "FAILURE"

                print(f"WARNING: {error_msg}. Retrying in {retry_delay}s...")
                time.sleep(retry_delay)

        return "ERROR: Loop finished without success.", "FAILURE" # Should not be reached


# define review call
def standardise_review(
    client: LLMClient,
    system_prompt_processing: str,
    user_prompt_processing: str,
    id_variable: str,
    temperature: float = 0,
    max_out_tokens: int = 4000,
    model: str = "gpt-4o-2024-05-13"
) -> Dict[str, str]:
    """
    Function to call the LLMClient's unified API/Sim call method.
    """

    # Use the LLMClient's call method
    generated_text, status = client.call(
        model=model,
        system_prompt=system_prompt_processing,
        user_prompt=user_prompt_processing,
        temperature=temperature,
        max_tokens=max_out_tokens,
    )

    return {
        'id': id_variable,
        'standardised_review_text': generated_text,
        'standardization_status': status
    }

In [ ]:
# @title Run simulation

def simulate_parameter_combo(
    combo: ParamCombo,
    targets: pd.DataFrame,
    client: LLMClient,
) -> pd.DataFrame:
    """Runs the standardization over ALL reviews in the targets DataFrame."""

    records = []

    SYSTEM_PROMPT = (
        "You are a neutral review standardizer. Convert the user-provided review into an objective, "
        "three-sentence summary. Sentence 1: Summarize the positive points. "
        "Sentence 2: Summarize the negative points. Sentence 3: Summarize the overall sentiment."
    )

    print(f"Running standardization on {len(targets)} reviews...")

    # loop over the dataframe rows
    for idx, row in tqdm(targets.iterrows(), total=len(targets), desc="Processing Reviews"):

        review_text = row['human_review1']
        review_id = row['review_id']

        # execution
        try:
            result = standardise_review(
                client=client,
                system_prompt_processing=SYSTEM_PROMPT,
                user_prompt_processing=review_text,
                id_variable=review_id,
                temperature=combo.temperature,
                model=combo.model
            )

            records.append({
                "review_id": review_id,
                "standardised_review_text": result["standardised_review_text"],
                "standardization_status": result["standardization_status"]
            })

        except Exception as e:
            # in case of error it is looked
            print(f"\n[ERROR] Review {review_id} -> {type(e).__name__}: {e}", flush=True)
            records.append({
                "review_id": review_id,
                "standardised_review_text": f"PROCESSING FAILED: {type(e).__name__}",
                "standardization_status": "FAILURE"
            })

    df_combo = pd.DataFrame.from_records(records)
    return df_combo


# execution of simulation
if __name__ == '__main__':
    targets_data = {
        'review_id': [f'R{i:04}' for i in range(1, 6)],
        'human_review1': [
            "The customer service was non-existent, but the product quality was outstanding.",
            "Average experience overall. The staff was neither helpful nor rude.",
            "Absolutely phenomenal. I wish I had bought two!",
            "Major flaws in the user interface; it crashed three times during my first hour.",
            "Good value for the price, though shipping was delayed by a week."
        ]
    }
    targets = pd.DataFrame(targets_data)

    # define setup
    combo = ParamCombo(
        model=LLM_MODEL,
        temperature=TEMP_SETTING
    )

    # check setup
    print(f"\n[RUN] Starting simple experiment: {combo.model} @ T={combo.temperature}", flush=True)

    # initiate client
    client = LLMClient(simulate=SIMULATE)

    try:
        # call
        df_results = simulate_parameter_combo(combo, targets, client=client)

        # save
        df_results.to_csv(RESULTS_PATH, index=False)

        # check
        print(f"\nSuccessfully processed and saved {len(df_results)} rows to {RESULTS_PATH}.")
        final_df = df_results

    # message error
    except Exception as e:
        print(f"\n[FATAL ERROR DURING RUN] {type(e).__name__}: {e}", flush=True)
        final_df = None

    # final status message
    if final_df is not None and not final_df.empty:
        print("\n--- Processed Results Preview ---")
        cols = ['review_id', 'standardised_review_text', 'standardization_status']
        print(final_df[cols].head().to_markdown(index=False))


In [ ]:
# @title Client Setup

# function for the simple API call
def standardise_review(system_prompt_processing: str, user_prompt_processing: str, id_variable: str, temperature: float = 0, max_out_tokens: int = 4000, model: str = "gpt-4o-2024-05-13"): # defaults as Xu et al.
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt_processing},
                {"role": "user", "content": user_prompt_processing}
            ],
            temperature=temperature,
            max_tokens=max_out_tokens
        )

        # Extract the review text from the response
        generated_text = response.choices[0].message.content
        return {
            'id': id_variable,
            'standardised_review_text': generated_text
        }

    except Exception as e:
        print(f"An API error occurred: {e}")
        return {'id': id_variable,
                'standardised_review_text': f"An API error occurred: {e}"
                }

## Summary of author stated strengths and weaknesses

**System prompt**

You are given a paper submission for a top-tier Machine Learning conference. Your goal is to identify and list the strengths and weaknesses that the paper claims about itself. This task requires careful reading of the paper. Please follow these steps to complete the task: 1. Carefully read the entire paper submission. As you read, identify instances where the authors mention strengths or positive aspects of their research, methodology, results, or contributions. These are the strengths claimed by the paper. Also, identify instances where the authors mention limitations, weaknesses, or areas for future improvement in their work. These are the weaknesses claimed by the paper. 2. Compile your findings into two separate lists: one for strengths and one for weaknesses. 3. For each list, write each point on a separate line, keeping it concise. Add an extra blank line between each point for clarity. 4. Format your output as follows: ¡strengths claimed by the paper¿ [List each strength claimed by the paper in separate lines, with an extra blank line between each point] ¡/strengths claimed by the paper¿ ¡weaknesses claimed by the paper¿ [List each weakness claimed by the paper in separate lines, with an extra blank line between each point] ¡/weaknesses claimed by the paper¿ Important: Focus only on the strengths and weaknesses that the paper claims about itself. Do not include your own evaluation or opinion of the paper’s merits or shortcomings. Do not include the strengths and weaknesses of the baseline. Your task is to report what the authors themselves have stated about their work’s strengths and limitations.

**User prompt**

{Full paper in Text Format}

## Judgement prediction

**System Prompt**

You are the second reviewer for a scientific paper. You are given the abstract of the paper and a list of review judgments from the first reviewer, starting with ’The reviewer appreciates/criticizes/questions/suggests’. Your task is to provide your own judgments of the paper based on the given materials. You should create a separate line for each judgment you have, starting with ’The reviewer appreciates/criticizes/questions/suggests’. Ensure your judgments are concise, excluding specific details about the paper’s content.

**User Prompt**

[Abstract of the paper]

{Abstract of the Paper if the metric is GEM-S, and “Not Available” if the metric is GEM}

[Review judgments from the first reviewer]

{Preprocessed Review Judgments (x) for conditional probability, and “Not Available” for marginal probability}

**Forced LLM Output**

{Preprocessed Review Judgments (y)}